# Lekcija 10 - AI agenti v produkciji

V tej lekciji se boste naučili **produkcijskih vzorcev** za AI agente z uporabo Microsoft Agent Framework (Python). Pokrivamo:

- **Opazljivost** — dodajanje merjenja časa in beleženja interakcij agentov
- **Vrednotenje** — uporaba ocenjevalnega agenta za oceno kakovosti odgovorov
- **Upravljanje stroškov** — strategije za optimizacijo porabe tokenov in izbiro modela

Scenarij je **potovalni agent**, ki pomaga uporabnikom pri načrtovanju potovanj, z dodanim spremljanjem in vrednotenjem.


## Nastavitev


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## Premisleki za produkcijo

Premik AI agentov iz prototipov v produkcijo zahteva skrbno pozornost trem stebrom:

1. **Opazljivost** — Potrebujete pregled nad tem, kaj agent počne, koliko časa traja in katera orodja kliče. Brez sledenja in beleženja je odpravljanje napak v produkciji skoraj nemogoče.

2. **Vrednotenje** — Avtomatizirane preverbe kakovosti zagotavljajo, da agentovi odgovori skozi čas ostajajo natančni, popolni in koristni. Ocenjevalni agent lahko ocenjuje odgovore glede na določena merila.

3. **Upravljanje stroškov** — Uporaba tokenov neposredno vpliva na stroške. Strategije, kot so optimizacija pozivov, izbira modela in predpomnjenje, pomagajo ohranjati stroške pod nadzorom, ne da bi pri tem žrtvovali kakovost.


## Ustvarjanje opaznega agenta

Opredelimo orodja za potovanje in obkrožimo klic agenta z merjenjem časa, da lahko spremljamo zakasnitev. V produkcijskem okolju bi integrirali OpenTelemetry ali podoben sistem za sledenje.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Vzorci ocenjevanja

Pogost vzorec v produkciji je uporaba drugega agenta kot **ocenjevalec**. Ocenjevalec ovrednoti odziv primarnega agenta glede na vnaprej določena merila, kot so popolnost, natančnost in koristnost.

To omogoča:
- Avtomatizirane kontrole kakovosti, preden odgovori dosežejo uporabnike
- Odkritje regresij, ko se spremenijo pozivi ali modeli
- Neprekinjeno spremljanje uspešnosti agenta skozi čas


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Strategije upravljanja stroškov

Nadzorovanje stroškov je ključnega pomena za AI agente v produkciji. Tukaj so ključne strategije:

| Strategija | Opis |
|---|---|
| **Optimizacija pozivov** | Ohranjajte sistemska navodila jedrnata. Odstranite odvečen kontekst, da zmanjšate število vhodnih tokenov. |
| **Izbira modela** | Uporabljajte manjše, cenejše modele (npr. GPT-4o-mini) za preprosta opravila, kot so klasifikacija ali izvleček, večje modele pa rezervirajte za kompleksno sklepanje. |
| **Predpomnjenje** | Predpomnite rezultate orodij in pogoste poizvedbe, da se izognete odvečnim klicem API. |
| **Proračuni tokenov** | Nastavite omejitve `max_tokens`, da preprečite nepričakovano dolge odgovore. |
| **Serijsko obdelovanje** | Združite več uporabniških poizvedb v en sam API klic, kjer je to mogoče. |

V praksi se dobro obnese večstopenjski pristop: enostavne zahteve usmerite k hitremu, poceni modelu, zahtevnejše pa eskalirajte le k zmogljivejšemu.


## Povzetek

V tej lekciji ste se naučili, kako:

1. **Dodati opazljivost** v interakcije agentov z merjenjem časa in beleženjem, s tem položiti temelje za sledenje in spremljanje.
2. **Oceniti odzive agenta** samodejno z uporabo ocenjevalnega agenta, ki oceni popolnost, natančnost in koristnost.
3. **Upravljati stroške** prek optimizacije navodil, izbire modela, predpomnjenja in proračunov žetonov.

Ti produkcijski vzorci pomagajo zagotoviti, da so vaši AI agenti zanesljivi, merljivi in stroškovno učinkoviti v večjem obsegu.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Izjava o omejitvi odgovornosti**:
Ta dokument je bil preveden z uporabo storitve za prevajanje z umetno inteligenco [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, upoštevajte, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v izvirnem jeziku velja za verodostojen vir. Za pomembne informacije priporočamo strokovni prevod, opravljen s strani profesionalnega prevajalca. Ne odgovarjamo za morebitne nesporazume ali napačne razlage, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
